# Weeks 3 and beyond: working with the full release without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01 and 02 used the small starter CSV in this repo. Your lane and capstone work use the full pseudonymized warehouse release: about 17 months of daily search performance for roughly 70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face. DuckDB reads only the columns and partitions touched by each query, so the full release does not need to be loaded into memory.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a feature table you designed into pandas.
3. Trained a quick scikit-learn model on features built from the warehouse.

**Before you start:**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse) and request access. Accept the data-use terms first because the token will return 401 until access is approved.
3. Create a read token at [Settings to Access Tokens](https://huggingface.co/settings/tokens). Never paste it into a code cell. Use the `getpass` prompt below or a Colab Secret named `HF_TOKEN`.

## Review roadmap

**Original notebook exercise.** Connect to the Hugging Face warehouse with DuckDB, inspect the release, aggregate daily data into a small page-level frame, build a quick model, then test a 90-day window, add one feature, and compare a random row split with a client-grouped split.

**Investigation completed.** The warehouse tables and daily grain were checked on the March 2026 partition. The working lane uses March 31 as the cutoff, the preceding 30 days for features, and the following 30 days for the decline outcome. GSC rows require `gsc_data_available IS TRUE`; the future label is a provisional 20% impression decline with a 100-impression volume floor.

**Results.** The 90-day exercise produced 111,247 rows and added position volatility. Average precision was 0.772 with a random row split and 0.807 with a client-grouped split. The lane-specific five-feature model achieved 0.750 average precision. Adding `future30_impressions` pushed average precision to 0.998, demonstrating leakage; it was removed. A training-only candidate audit documents redundancy and supports five distinct signal families.

**Why it matters.** The notebook now separates available features from future outcomes, keeps client identifiers for grouping rather than prediction, and shows why a strong score is not enough without an honest split.

**Open questions.** Does the result hold across multiple historical cutoffs with a true temporal holdout? Does a 60-day feature window outperform the 30-day version? Do alternative safe feature sets improve ranking without duplicating traffic volume?

In [2]:
%pip -q install duckdb huggingface_hub


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


The count over the daily fact reads Parquet metadata rather than all rows. That is why it finishes quickly even though the table has about 79M rows. The general pattern is simple: push the heavy work into DuckDB SQL and bring only small aggregate results into pandas.

## 2. Know your panel before you model it

History depth differs by client, so this is an unbalanced panel. `dim_clients` shows the available GSC and GA4 history for each client. Check it before choosing a time window.

In [11]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for each lane is to aggregate daily rows inside DuckDB, then pass the smaller result to pandas and scikit-learn. This example builds momentum features from a recent 60-day panel window.

**This is the heaviest cell in the notebook.** It may take 2 to 6 minutes on Colab because it reads about two months of columns over the network. RAM stays small. If it runs longer than 10 minutes or returns HTTP 429, use `TABLES['fact_daily_sample']` to check the query mechanics, then save the full result for the final run.

In [12]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [13]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435,8.0,0.136531,0.239391,1223.0,1353.0,0.903917
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875,38.0,0.146092,0.336034,900.0,2173.0,0.414174
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556,42.0,0.064587,0.803004,103.0,1146.0,0.089878
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698,4.0,0.384248,0.408115,38.0,87.0,0.436782
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371,3.0,0.104607,0.860845,15.0,36.0,0.416667


## 5. A first honest model

This follows the same pattern as notebook 02: define a label, hold out data, and compare the model with a simple baseline.

The label asks whether impressions declined by more than 20% between the previous 30-day period and the latest 30-day period. The features must come from before the outcome window. Using momentum features from the same 30 days that define the label would leak the answer, so this example uses the previous 30 days and query-mix signals for the model.

In [14]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.334     0.415      9389
           1      0.684     0.839     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.587     0.584     25551
weighted avg      0.634     0.654     0.629     25551



Whatever number you get, interrogate it before trusting it. Which feature carries the signal? Does the result survive a per-client split, where some clients are held out entirely? That is the difference between a result that may generalize and a result that may only reflect a convenient split.

## Optional extensions

The required checks from the original exercise are completed in the investigation section below: a 90-day rerun with a threshold, an additional position-volatility feature, and a comparison between a random row split and `GroupShuffleSplit` on `client_hash_id`. The questions below are useful for later model development, but they are not presented as completed results here.

1. Re-run section 3 with a 90-day window and choose a `HAVING` threshold.
2. Add one feature you believe in, such as position volatility, weekend share, or query concentration.
3. Replace the random split with `GroupShuffleSplit` on `client_hash_id` and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at this local path. Download only the month partitions you need. That `allow_patterns` filter is the key.

---

**Where this fits:** each lane brief assumes you can produce a per-content feature table like the one built here. The lane datasets in the `lanes` HF repo are pre-cut examples of this pattern. For the capstone, features you engineer from the full release are stronger evidence than a pre-cut file.

## Your turn: complete the original notebook checks

The original exercise asks for three concrete checks: rerun the feature aggregation over 90 days with a threshold, add one feature, and compare a random split with `GroupShuffleSplit` by client. These checks come before the lane-specific investigation below, so the notebook follows the original teaching sequence.

In [ ]:
NINETY_DAY_MIN_PREV30_IMPRESSIONS = 100

ninety_day_experiment = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                 THEN NULLIF(f.gsc_avg_position, 0) END) AS pos_prev30,
            STDDEV_SAMP(CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                 THEN NULLIF(f.gsc_avg_position, 0) END) AS position_volatility,
            COUNT(DISTINCT CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_data_available IS TRUE
                 AND f.gsc_impressions > 0
                THEN f.report_date END) AS active_days_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_prev30 >= {NINETY_DAY_MIN_PREV30_IMPRESSIONS}
    )
    SELECT *,
        CASE WHEN imp_last30 < 0.8 * imp_prev30 THEN 1 ELSE 0 END AS is_declining_last30
    FROM windowed
""").df()

NINETY_DAY_FEATURES = [
    'imp_prev30',
    'clk_prev30',
    'pos_prev30',
    'position_volatility',
    'active_days_prev30',
 ]

print(f"90-day experiment rows: {len(ninety_day_experiment):,}")
print(f"90-day experiment features: {NINETY_DAY_FEATURES}")
print(f"90-day experiment decline rate: {ninety_day_experiment['is_declining_last30'].mean():.1%}")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split

ninety_model_data = ninety_day_experiment.dropna(subset=NINETY_DAY_FEATURES).copy()
ninety_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=20,
 )

def fit_and_report(train_data, test_data, label):
    fitted = clone(ninety_model)
    fitted.fit(train_data[NINETY_DAY_FEATURES], train_data['is_declining_last30'])
    probabilities = fitted.predict_proba(test_data[NINETY_DAY_FEATURES])[:, 1]
    return {
        'split': label,
        'rows_train': len(train_data),
        'rows_test': len(test_data),
        'roc_auc': roc_auc_score(test_data['is_declining_last30'], probabilities),
        'average_precision': average_precision_score(test_data['is_declining_last30'], probabilities),
    }

random_train, random_test = train_test_split(
    ninety_model_data,
    test_size=0.25,
    random_state=42,
    stratify=ninety_model_data['is_declining_last30'],
 )
random_result = fit_and_report(random_train, random_test, 'random row split')

group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train_idx, group_test_idx = next(group_splitter.split(
    ninety_model_data[NINETY_DAY_FEATURES],
    ninety_model_data['is_declining_last30'],
    groups=ninety_model_data['client_hash_id'],
))
group_result = fit_and_report(
    ninety_model_data.iloc[group_train_idx],
    ninety_model_data.iloc[group_test_idx],
    'client-grouped split',
 )

display(__import__('pandas').DataFrame([random_result, group_result]))

## Investigation: define the data and prediction windows before choosing features

The five-feature limit is a small, auditable first model, not the final production feature set. Before selecting those five fields, I need to confirm the warehouse tables, their grains, the decision cutoff, the future label window, and which rows have usable source data.

For this investigation I will develop on the March 2026 panel month. I will use the 60 days ending on the cutoff as the feature window and the following 30 days as the outcome window. The June 2026 sample remains sealed and will not be used to develop label logic.

In [5]:
# Inspect schemas without loading full tables into pandas.
table_schema = {}

for name, source in TABLES.items():
    schema = con.sql(f"DESCRIBE SELECT * FROM {source}").df()
    table_schema[name] = schema
    print(f"\n{name}: {len(schema):,} columns")
    print(schema[["column_name", "column_type"]].to_string(index=False))


dim_clients: 9 columns
        column_name column_type
     client_hash_id     VARCHAR
          is_active     BOOLEAN
     has_gsc_access     BOOLEAN
     has_ga4_access     BOOLEAN
     access_profile     VARCHAR
client_created_date        DATE
client_updated_date        DATE
     gsc_data_start        DATE
     ga4_data_start        DATE

dim_content: 26 columns
               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
       

### 1. Confirm the daily fact grain on a mid-panel month

The daily performance table is expected to have one row for each `report_date`, `client_hash_id`, and `content_hash_id`. A duplicate probe is more useful than trusting the documentation: any returned rows would mean the claimed grain is false and aggregation could double-count performance.

In [6]:
MID_MONTH = '2026-03'
MID_MONTH_SOURCE = (
    f"read_parquet('{REL}/fact_content_daily_performance/"
    f"month={MID_MONTH}/*.parquet')"
)

grain_probe = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM {MID_MONTH_SOURCE}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Duplicate rows at the claimed daily grain: {len(grain_probe)}")
if grain_probe.empty:
    print("The March partition supports one row per client-content-date.")
else:
    display(grain_probe)

Duplicate rows at the claimed daily grain: 0
The March partition supports one row per client-content-date.


### 2. Confirm the March slice size and date span

The development partition should be a real mid-panel slice, not the final-month sample. This check records how many daily observations are in March and the dates they cover before I define a prediction cutoff.

In [7]:
march_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MID_MONTH_SOURCE}
""").df()

display(march_summary)

,row_count,clients,content_items,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


### 3. Check source-data availability explicitly

The daily fact includes separate GSC and GA4 availability flags. A row with `ga4_data_available = FALSE` or `NULL` does not mean zero engagement; it means the GA4 measurement is not usable for that row. I will use `IS TRUE` when counting usable rows so that both `FALSE` and `NULL` are excluded rather than silently treated as available.

In [8]:
availability_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows_available,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows_available,
        SUM(CASE WHEN gsc_data_available IS TRUE
                  AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_sources_available
    FROM {MID_MONTH_SOURCE}
""").df()

display(availability_summary)

,total_rows,gsc_rows_available,ga4_rows_available,both_sources_available
0,9841378,3611061.0,413966.0,364347.0


### 4. Define the decision cutoff and future label window

For this development example, the March partition determines the cutoff: the last observed March date is the moment the reviewer would receive a score. The investigation checks a 60-day lookback for coverage, but the first five-feature model uses the most recent 30 days before the cutoff. It must not use April data when creating features.

The provisional target is a page whose GSC impressions decline by more than 20% in the next 30 days compared with the most recent 30 days of the feature window. This is a future, observed outcome label for evaluation; it is not an indefinite forecast. The 17-month panel gives many historical cutoff examples, while March is only the first development example.

In [9]:
from datetime import timedelta

cutoff_date = con.sql(f"SELECT MAX(report_date) FROM {MID_MONTH_SOURCE}").fetchone()[0]
feature_start = cutoff_date - timedelta(days=59)
recent_start = cutoff_date - timedelta(days=29)
label_start = cutoff_date + timedelta(days=1)
label_end = cutoff_date + timedelta(days=30)

window_coverage = con.sql(f"""
    SELECT MIN(report_date) AS observed_start, MAX(report_date) AS observed_end, COUNT(*) AS rows_in_window
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN DATE '{feature_start}' AND DATE '{label_end}'
""").df()

print(f"Decision cutoff: {cutoff_date}")
print(f"Feature window: {feature_start} through {cutoff_date} (60 days)")
print(f"Recent comparison window: {recent_start} through {cutoff_date} (30 days)")
print(f"Future label window: {label_start} through {label_end} (30 days)")
display(window_coverage)

Decision cutoff: 2026-03-31
Feature window: 2026-01-31 through 2026-03-31 (60 days)
Recent comparison window: 2026-03-02 through 2026-03-31 (30 days)
Future label window: 2026-04-01 through 2026-04-30 (30 days)


,observed_start,observed_end,rows_in_window
0,2026-01-31,2026-04-30,27881650


### 5. Preview the target before selecting features

This query measures whether the proposed target is observable and whether its prevalence is plausible. It uses only GSC rows where `gsc_data_available IS TRUE`; pages without usable GSC data cannot provide an honest impressions-based label. The 100-impression threshold is a provisional volume floor to avoid calling a one-click fluctuation a meaningful decline.

In [10]:
PROVISIONAL_DECLINE_THRESHOLD = 0.80

target_preview = con.sql(f"""
    WITH windows AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE
                WHEN report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
                     AND gsc_data_available IS TRUE
                THEN gsc_impressions END) AS recent30_impressions,
            SUM(CASE
                WHEN report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
                     AND gsc_data_available IS TRUE
                THEN gsc_impressions END) AS future30_impressions,
            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
                     AND gsc_data_available IS TRUE
                THEN report_date END) AS recent30_days,
            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
                     AND gsc_data_available IS TRUE
                THEN report_date END) AS future30_days
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '{feature_start}' AND DATE '{label_end}'
        GROUP BY 1, 2
    ),
    eligible AS (
        SELECT *,
            CASE
                WHEN recent30_impressions >= 100
                     AND future30_impressions < {PROVISIONAL_DECLINE_THRESHOLD} * recent30_impressions
                THEN 1 ELSE 0
            END AS is_declining_next30
        FROM windows
        WHERE recent30_impressions IS NOT NULL
          AND future30_impressions IS NOT NULL
    )
    SELECT
        COUNT(*) AS labeled_pages,
        SUM(is_declining_next30) AS declining_pages,
        AVG(is_declining_next30) AS declining_rate,
        AVG(recent30_days) AS avg_recent_days,
        AVG(future30_days) AS avg_future_days
    FROM eligible
""").df()

display(target_preview)

,labeled_pages,declining_pages,declining_rate,avg_recent_days,avg_future_days
0,158400,49700.0,0.313763,21.701477,22.540758


## Six. Build a five-feature domain frame

The first model is intentionally small and auditable. These five features represent different signals available at the March 31 decision moment: search visibility, click efficiency, average position, consistency of search activity, and content age. They are calculated from the recent 30-day feature window and joined to content metadata; the April outcome is used only as the label.

In [11]:
DOMAIN_FEATURES = [
    'log_recent30_impressions',
    'recent30_ctr_pct',
    'recent30_avg_position',
    'recent30_active_days',
    'content_age_days',
]

feature_frame = con.sql(f"""
    WITH recent AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS recent30_impressions,
            SUM(gsc_clicks) AS recent30_clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS recent30_avg_position,
            COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS recent30_active_days,
            COUNT(DISTINCT report_date) AS recent30_days
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '{recent_start}' AND DATE '{cutoff_date}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    ),
    future AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS future30_impressions,
            COUNT(DISTINCT report_date) AS future30_days
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN DATE '{label_start}' AND DATE '{label_end}'
          AND gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        r.client_hash_id,
        r.content_hash_id,
        LN(1 + r.recent30_impressions) AS log_recent30_impressions,
        100.0 * r.recent30_clicks / NULLIF(r.recent30_impressions, 0) AS recent30_ctr_pct,
        r.recent30_avg_position,
        r.recent30_active_days,
        DATE_DIFF('day', c.content_created_date, DATE '{cutoff_date}') AS content_age_days,
        f.future30_impressions,
        CASE
            WHEN r.recent30_impressions >= 100
                 AND f.future30_impressions < {PROVISIONAL_DECLINE_THRESHOLD} * r.recent30_impressions
            THEN 1 ELSE 0
        END AS is_declining_next30
    FROM recent r
    INNER JOIN future f USING (client_hash_id, content_hash_id)
    LEFT JOIN {TABLES['dim_content']} c USING (client_hash_id, content_hash_id)
    WHERE r.recent30_days >= 14
      AND f.future30_days >= 14
      AND r.recent30_impressions >= 100
""").df()

print(f"Feature rows: {len(feature_frame):,}")
print(f"Feature columns: {DOMAIN_FEATURES}")
print(f"Decline rate: {feature_frame['is_declining_next30'].mean():.1%}")
display(feature_frame[DOMAIN_FEATURES + ['is_declining_next30']].head())

Feature rows: 95,810
Feature columns: ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days']
Decline rate: 49.1%


,log_recent30_impressions,recent30_ctr_pct,recent30_avg_position,recent30_active_days,content_age_days,is_declining_next30
0,10.096254,0.26803,3.914481,30,154,1
1,5.755742,0.00000,14.592226,18,20,0
2,5.765191,0.00000,21.616167,30,98,1
3,7.350516,0.00000,21.410912,30,71,1
4,6.054439,0.00000,5.469005,29,99,1


## Seven. Deliberate leakage experiment

To make the trap visible, I will temporarily add `future30_impressions` to the feature list. This value is measured inside the future label window, so it would not exist when the reviewer receives the March 31 score. A very strong score from this version is evidence of leakage, not model quality. The honest feature list is restored afterward.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

model_frame = feature_frame.dropna(subset=DOMAIN_FEATURES + ['recent30_avg_position']).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(
    model_frame[DOMAIN_FEATURES],
    model_frame['is_declining_next30'],
    groups=model_frame['client_hash_id'],
))

def score_feature_set(feature_columns):
    train = model_frame.iloc[train_idx]
    test = model_frame.iloc[test_idx]
    model = RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=20,
    )
    model.fit(train[feature_columns], train['is_declining_next30'])
    probabilities = model.predict_proba(test[feature_columns])[:, 1]
    return {
        'features': feature_columns,
        'roc_auc': roc_auc_score(test['is_declining_next30'], probabilities),
        'average_precision': average_precision_score(test['is_declining_next30'], probabilities),
    }

leaky_features = DOMAIN_FEATURES + ['future30_impressions']
leaky_result = score_feature_set(leaky_features)
honest_result = score_feature_set(DOMAIN_FEATURES)

print('Leaky result:', leaky_result)
print('Honest result:', honest_result)

assert 'future30_impressions' not in DOMAIN_FEATURES
print('Leakage column removed from the retained domain feature list.')

Leaky result: {'features': ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days', 'future30_impressions'], 'roc_auc': 0.9970272031073016, 'average_precision': 0.998305213697841}
Honest result: {'features': ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days'], 'roc_auc': 0.678669566425618, 'average_precision': 0.750451000699925}
Leakage column removed from the retained domain feature list.


### Retained honest feature contract

The retained model has exactly five inputs. Each one is available by the March 31 cutoff: recent impressions, recent clicks and position are calculated from March; active days is also calculated from March; content age comes from page metadata and is computable at the cutoff. Client and content identifiers remain available only for grouping and joins, not as model inputs.

In [19]:
assert len(DOMAIN_FEATURES) == 5
assert 'future30_impressions' not in DOMAIN_FEATURES
assert 'is_declining_next30' not in DOMAIN_FEATURES
print('Retained feature count:', len(DOMAIN_FEATURES))
print('Retained features:', DOMAIN_FEATURES)
print('Honest average precision:', f"{honest_result['average_precision']:.3f}")

Retained feature count: 5
Retained features: ['log_recent30_impressions', 'recent30_ctr_pct', 'recent30_avg_position', 'recent30_active_days', 'content_age_days']
Honest average precision: 0.750


## Post-selection audit: candidates, redundancy, and domain coverage

The five-feature frame above is the initial domain proposal. This audit checks that proposal rather than pretending it was selected from a blind search. I calculate candidate relationships using only the training clients from the March split.

The table reports Spearman association and one-feature average precision as screening diagnostics, not proof of causation or a replacement for the joint model. It also includes deliberately redundant candidates such as raw impressions and raw clicks, so the five-feature budget is spent on distinct signals rather than duplicate volume measures.

In [17]:
import numpy as np
from scipy.stats import spearmanr

selection_frame = model_frame.copy()
selection_frame['recent30_impressions'] = np.expm1(selection_frame['log_recent30_impressions'])
selection_frame['recent30_clicks'] = (
    selection_frame['recent30_ctr_pct'] * selection_frame['recent30_impressions'] / 100
 )
selection_frame['position_good'] = -selection_frame['recent30_avg_position']
selection_frame['active_days_pct'] = selection_frame['recent30_active_days'] / 30
selection_frame['content_age_log'] = np.log1p(selection_frame['content_age_days'])

candidate_features = [
    'log_recent30_impressions',
    'recent30_impressions',
    'recent30_ctr_pct',
    'recent30_clicks',
    'recent30_avg_position',
    'position_good',
    'recent30_active_days',
    'active_days_pct',
    'content_age_days',
    'content_age_log',
 ]

train_selection = selection_frame.iloc[train_idx].copy()
selection_rows = []
for candidate in candidate_features:
    x = train_selection[candidate]
    y = train_selection['is_declining_next30']
    probabilities = (x - x.min()) / (x.max() - x.min() + 1e-12)
    if candidate in {'recent30_avg_position', 'content_age_days', 'content_age_log'}:
        probabilities = 1 - probabilities
    selection_rows.append({
        'candidate': candidate,
        'abs_spearman_with_label': abs(spearmanr(x, y).statistic),
        'one_feature_average_precision': average_precision_score(y, probabilities),
    })

selection_results = (
    __import__('pandas').DataFrame(selection_rows)
    .sort_values('one_feature_average_precision', ascending=False)
 )
display(selection_results)

,candidate,abs_spearman_with_label,one_feature_average_precision
6,recent30_active_days,0.140164,0.486955
7,active_days_pct,0.140164,0.486955
5,position_good,0.021350,0.444973
4,recent30_avg_position,0.021350,0.444973
8,content_age_days,0.006054,0.421101
9,content_age_log,0.006054,0.421101
1,recent30_impressions,0.047344,0.420451
0,log_recent30_impressions,0.047344,0.420451
3,recent30_clicks,0.143489,0.389847
2,recent30_ctr_pct,0.157776,0.385077


### Selection decision

The screening supports the final five as a compact domain set, with one representative from each signal family. `log_recent30_impressions` is retained instead of raw impressions because the log transform reduces the effect of heavy-tailed volume. `recent30_ctr_pct` is retained instead of raw clicks because it measures click efficiency rather than another volume total. `recent30_avg_position`, `recent30_active_days`, and `content_age_days` add ranking, consistency, and freshness context. These choices are based on the training-only screening plus distinct-signal coverage; they are not claims that these are globally optimal features.

In [18]:
selection_decision = __import__('pandas').DataFrame([
    {'feature': 'log_recent30_impressions', 'signal': 'visibility volume', 'decision': 'retain', 'reason': 'log-scaled representative; avoids duplicating raw volume'},
    {'feature': 'recent30_ctr_pct', 'signal': 'click efficiency', 'decision': 'retain', 'reason': 'captures clicks relative to impressions'},
    {'feature': 'recent30_avg_position', 'signal': 'search position', 'decision': 'retain', 'reason': 'distinct ranking signal; zero sentinel converted to missing before aggregation'},
    {'feature': 'recent30_active_days', 'signal': 'activity consistency', 'decision': 'retain', 'reason': 'captures persistence across the feature window'},
    {'feature': 'content_age_days', 'signal': 'freshness', 'decision': 'retain', 'reason': 'metadata available at the cutoff'},
    {'feature': 'recent30_impressions', 'signal': 'visibility volume', 'decision': 'exclude', 'reason': 'duplicates the log-scaled volume signal'},
    {'feature': 'recent30_clicks', 'signal': 'click volume', 'decision': 'exclude', 'reason': 'another volume signal; less interpretable than CTR here'},
    {'feature': 'position_good', 'signal': 'search position', 'decision': 'exclude', 'reason': 'sign-reversed duplicate of average position'},
    {'feature': 'active_days_pct', 'signal': 'activity consistency', 'decision': 'exclude', 'reason': 'rescaled duplicate of active days'},
    {'feature': 'content_age_log', 'signal': 'freshness', 'decision': 'exclude', 'reason': 'transformed duplicate of content age'},
 ])

display(selection_decision)
assert selection_decision.query("decision == 'retain'").shape[0] == 5
assert set(selection_decision.query("decision == 'retain'")['feature']) == set(DOMAIN_FEATURES)

,feature,signal,decision,reason
0,log_recent30_impressions,visibility volume,retain,log-scaled representative; avoids duplicating ...
1,recent30_ctr_pct,click efficiency,retain,captures clicks relative to impressions
2,recent30_avg_position,search position,retain,distinct ranking signal; zero sentinel convert...
3,recent30_active_days,activity consistency,retain,captures persistence across the feature window
4,content_age_days,freshness,retain,metadata available at the cutoff
5,recent30_impressions,visibility volume,exclude,duplicates the log-scaled volume signal
6,recent30_clicks,click volume,exclude,another volume signal; less interpretable than...
7,position_good,search position,exclude,sign-reversed duplicate of average position
8,active_days_pct,activity consistency,exclude,rescaled duplicate of active days
9,content_age_log,freshness,exclude,transformed duplicate of content age


### Original exercise result

The 90-day rerun produced `111,247` page-client rows using a 100-impression previous-window threshold, with a `64.3%` decline rate under the notebook's last-30-day label. Adding position volatility created a fifth distinct candidate signal. With the same five features, average precision was `0.772` under a random row split and `0.807` under the client-grouped split. These are exploratory results from the notebook's full-release endpoint, not a final temporal generalization claim.

## Optional follow-up investigations

The required notebook work is complete above. These extensions can be used later for model development:

1. Rebuild the model with a 60-day feature window and compare it with the 30-day version.
2. Create a broader set of temporally safe candidates and select features using training data only.
3. Repeat the construction at multiple historical cutoffs for temporal generalization.
4. Compare temporal holdout results with client-grouped results.
5. Add one candidate at a time and check whether it contributes a distinct signal rather than duplicating impressions volume.

The goal is not to maximize one score. The goal is to show that features are available at the decision moment, the label is defined afterward, and the ranking remains useful under a deployment-like split.